# H&N vs Lung – HCP Segmentation

**Approach (agreed with the team):** compare two HCP lists at the **J&J ID** level.

| List | Source |
|---|---|
| **H&N cohort** | `Rybrevant_HNSCC_HV_CLAIMS` (built by the LE/AI team) |
| **Rybrevant writers, last 52 weeks** | Medical claims (`rendering_npi`) + pharmacy claims (`prescriber_npi`, only for patients not in medical claims), rolled up to J&J ID through `dmn_cust` |

**Segments**

| Segment | In the H&N cohort | A Rybrevant writer (52 weeks) |
|---|---|---|
| **H&N only** | yes | no |
| **H&N + Lung** | yes | yes |
| **Lung / Rybrevant only** | no | yes |

- No diagnosis codes are used here.
- McKesson EMR (section 6) and Xponent retail sales (section 7) are shown as extra HCP-level sources.
- **DDD** (L1 outlet) and **Xponent** (L0 HCP) sales are rolled up to the L2/L3 account through OPAL (section 9).

| Level | Meaning |
|---|---|
| L0 | Physician / HCP (claims, Xponent, H&N cohort) |
| L1 | Outlet / hospital pharmacy (DDD) |
| L2 | Hospital / account |
| L3 | Larger account |
| L4–L5 | Enterprise health system |

Sections: 1 setup · 2 date window · 3 NPI → J&J ID · 4 H&N cohort · 5 Rybrevant writers · 6 McKesson · 7 Xponent · 8 segments · 9 accounts + DDD + Xponent · 10 save

## 1. Setup

### What this cell does: settings and data loading
- **All settings are at the top. Change them here only.**
  - `DATASETS`: Dataiku dataset names. Update them when Ravi delivers the full-period tables.
  - `END_DATE` / `WINDOW_WEEKS`: the 52-week window, ending 2026-08-31.
  - `INCLUDE_MCKESSON_AS_WRITER` / `INCLUDE_XPONENT_AS_WRITER`: whether McKesson or Xponent HCPs count as Rybrevant writers. Default **False**, because the team hasn't decided; section 8 shows every combination.
- Loads all tables and prints their row counts.
- `c("...")` is a small helper for column names with spaces or symbols, like `J&J ID`.

In [ ]:
import datetime as dt
import dataiku
import dataiku.spark as dkuspark
from pyspark.sql import SparkSession, SQLContext, Window
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()
sqlContext = SQLContext(spark.sparkContext)

# ---------------- settings ----------------
DATASETS = {
    "hn_cohort": "Rybrevant_HNSCC_HV_CLAIMS",
    "mx":        "MX_RYB_CLAIMS_copy",
    "rx":        "RYB_LINK_CLAIMS_copy",
    "dmn":       "dmn_cust_copy",
    "opal":      "OPAL_Affiliation_Levels",
    "ddd":       "RYB_DDD_Sales_copy",
    "mckesson":  "Mckesson_GPO_DATA_copy",
    "xpon":      "RYB_Xpon_Sales_copy",
}
END_DATE = "2026-08-31"          # upper date cutoff
WINDOW_WEEKS = 52
INCLUDE_MCKESSON_AS_WRITER = False
INCLUDE_XPONENT_AS_WRITER = False

RYB_NDC = {   # NDC reference from RYB_HN_Data_Sources.csv (Rybrevant + Faspro only; Lazcluze excluded)
    "57894050100": "RYBREVANT", "57894050101": "RYBREVANT",
    "57894051001": "RYBREVANT FASPRO", "57894051401": "RYBREVANT FASPRO",
    "57894051501": "RYBREVANT FASPRO", "57894052201": "RYBREVANT FASPRO",
}
# -------------------------------------------

def load(name):
    return dkuspark.get_dataframe(sqlContext, dataiku.Dataset(name))

def c(name):
    """column whose name has spaces / symbols"""
    return F.col(f"`{name}`")

def clean(col):
    """trim; empty text -> null"""
    s = F.trim(col.cast("string"))
    return F.when(s != "", s)

df = {k: load(v) for k, v in DATASETS.items()}
for k, v in df.items():
    print(f"{k:10s} {DATASETS[k]:35s} rows={v.count():>12,}")

## 2. Date window

### What this cell does: define the 52-week window and check each table covers it
- Window = the 52 weeks (364 days) ending on `END_DATE`.
- For each dated table it shows the earliest and latest date, and whether the table **covers the full window**.
  - If a table starts after the window start, the 52-week counts will be incomplete until Ravi's full-period tables arrive.
- The window is applied to:
  - medical claims (`service_date`);
  - pharmacy claims (`fill_date`);
  - DDD and Xponent (`caln_dt`);
  - McKesson (`diagnosis_date`).
- The H&N cohort is used as delivered, since it has its own "2 years" definition.

In [ ]:
end_dt = dt.date.fromisoformat(END_DATE)
start_dt = end_dt - dt.timedelta(days=7 * WINDOW_WEEKS - 1)
START_DATE = start_dt.isoformat()
print(f"Window: {START_DATE} .. {END_DATE} ({WINDOW_WEEKS} weeks)")

for key, col in [("mx", "service_date"), ("rx", "fill_date"), ("ddd", "caln_dt"), ("xpon", "caln_dt"), ("mckesson", "diagnosis_date")]:
    r = df[key].agg(F.min(F.to_date(c(col))).alias("min"), F.max(F.to_date(c(col))).alias("max")).collect()[0]
    covers = r["min"] is not None and r["min"] <= start_dt and r["max"] >= end_dt - dt.timedelta(days=6)
    print(f"{key:9s} {col:15s} {r['min']} .. {r['max']}   covers full window: {'YES' if covers else 'NO'}")

def in_window(col):
    return F.to_date(c(col)).between(F.lit(START_DATE), F.lit(END_DATE))

## 3. NPI → J&J ID lookup (dmn_cust)

### What this cell does: build the NPI → J&J ID lookup
- From `dmn_cust`: `npi_id` → `jnj_id`, plus the customer type and primary specialty.
- **Rule (data-sources file):** if an NPI has several `jnj_id`s, keep the one with the latest `efftv_dt`.
- dmn_cust is already filtered upstream to Active, end_dt 2099-12-31, so it isn't filtered again.

In [ ]:
npi_jnj = (df["dmn"].select(clean(F.col("npi_id")).alias("npi"), clean(F.col("jnj_id")).alias("jnj_id"),
                            F.to_date("efftv_dt").alias("efftv_dt"),
                            F.col("cust_typ_nm").alias("dmn_cust_type"), F.col("prim_spclt_nm").alias("dmn_specialty"))
           .filter("npi IS NOT NULL AND jnj_id IS NOT NULL")
           .withColumn("rk", F.row_number().over(Window.partitionBy("npi").orderBy(F.desc_nulls_last("efftv_dt"), "jnj_id")))
           .filter("rk = 1").drop("rk", "efftv_dt")
           .cache())
print("NPIs in lookup:", f"{npi_jnj.count():,}")

## 4. H&N cohort (LE/AI team)

### What this cell does: profile the H&N cohort and roll it up to one row per HCP
- The cohort table has one row per HCP × mapped account, so an HCP can appear on several rows.
- **Profile:**
  - rows;
  - distinct NPIs and J&J IDs;
  - empty IDs;
  - HCPs on more than one row;
  - which account levels appear in `Mapped Account JNJ Level`.
- **Consistency check:** does the cohort's own `J&J ID` match the J&J ID that dmn_cust gives for the same NPI?
- **HCP key:**
  - the cohort's `J&J ID`;
  - if empty, the J&J ID from dmn_cust via the NPI;
  - if still empty, `NPI_<npi>`, so nobody is lost.
- Rolled up per HCP: name, NPI, specialty, practice setting, number of mapped accounts, and the largest values of the two patient-volume columns.

In [ ]:
coh = df["hn_cohort"].select(
    clean(c("NPI")).alias("npi"), clean(c("J&J ID")).alias("cohort_jnj_id"), clean(c("Name")).alias("name"),
    clean(c("Treating Patient Volume - HV (2 years)")).cast("double").alias("hn_treating_pts_2y"),
    clean(c("Origami-4 Eligible Patients (2 years)")).cast("double").alias("origami4_eligible_pts_2y"),
    clean(c("Mapped Account JNJ ID")).alias("cohort_account_id"), clean(c("Mapped Account JNJ Level")).alias("cohort_account_level"),
    clean(c("Mapped Account Name")).alias("cohort_account_name"),
    clean(c("Top Parent JNJ ID")).alias("top_parent_id"), clean(c("Top Parent Account Name")).alias("top_parent_name"),
    clean(c("Practice Settings")).alias("practice_setting"), clean(c("Final Specialty")).alias("cohort_specialty"))

coh.agg(F.count("*").alias("rows"), F.countDistinct("npi").alias("distinct_npi"),
        F.countDistinct("cohort_jnj_id").alias("distinct_jnj_id"),
        F.sum(F.col("npi").isNull().cast("int")).alias("empty_npi"),
        F.sum(F.col("cohort_jnj_id").isNull().cast("int")).alias("empty_jnj_id")).show()
coh.groupBy("cohort_account_level").count().orderBy("cohort_account_level").show()

# consistency: cohort J&J ID vs dmn_cust J&J ID for the same NPI
chk = coh.select("npi", "cohort_jnj_id").distinct().join(npi_jnj.select("npi", "jnj_id"), "npi", "left")
(chk.withColumn("check", F.when(F.col("jnj_id").isNull(), "NPI not in dmn_cust")
                         .when(F.col("cohort_jnj_id").isNull(), "cohort J&J ID empty")
                         .when(F.col("jnj_id") == F.col("cohort_jnj_id"), "same J&J ID")
                         .otherwise("different J&J ID"))
    .groupBy("check").count().orderBy(F.desc("count")).show(truncate=False))

coh_k = (coh.join(npi_jnj.select("npi", F.col("jnj_id").alias("dmn_jnj_id")), "npi", "left")
            .withColumn("hcp_key", F.coalesce("cohort_jnj_id", "dmn_jnj_id", F.concat(F.lit("NPI_"), "npi"))))
cohort_hcp = (coh_k.groupBy("hcp_key")
              .agg(F.first("npi", True).alias("cohort_npi"), F.first("name", True).alias("name"),
                   F.first("cohort_specialty", True).alias("cohort_specialty"),
                   F.first("practice_setting", True).alias("practice_setting"),
                   F.max("hn_treating_pts_2y").alias("hn_treating_pts_2y"),
                   F.max("origami4_eligible_pts_2y").alias("origami4_eligible_pts_2y"),
                   F.countDistinct("cohort_account_id").alias("cohort_n_accounts"),
                   F.count("*").alias("cohort_rows"))
              .withColumn("in_hn_cohort", F.lit(True))
              .cache())
print("H&N cohort HCPs:", f"{cohort_hcp.count():,}",
      "| on more than one row:", cohort_hcp.filter("cohort_rows > 1").count())

## 5. Rybrevant writers in the last 52 weeks

### What this cell does: build the list of Rybrevant writers, one row per HCP
- **Medical claims:**
  - Rybrevant / Faspro NDCs only, `service_date` in the window;
  - doctor = `rendering_npi` (Ravi);
  - one infusion = patient + day + NDC (Ravi).
- **Pharmacy claims:**
  - Rybrevant / Faspro NDCs only, `fill_date` in the window;
  - only patients **not** in the medical claims (Ravi);
  - doctor = `prescriber_npi`.
- Each NPI → J&J ID through dmn_cust. HCP key = J&J ID, or `NPI_<npi>` if not found.
- Per HCP: patients, infusions (medical), fills (pharmacy), and NPIs.
- Prints how many writer NPIs got a J&J ID.

In [ ]:
ndc_map = F.create_map(*[F.lit(x) for kv in RYB_NDC.items() for x in kv])
def ryb_product(ndc_col, name_col):
    name = F.upper(F.trim(F.col(name_col)))
    return F.coalesce(ndc_map[F.trim(F.col(ndc_col))], F.when(name.contains("RYBREVANT"), name))

mx_w = (df["mx"].withColumn("product", ryb_product("ndc", "product_name"))
          .filter(F.col("product").isNotNull() & in_window("service_date"))
          .select(F.concat_ws("|", "dv1", "dv2").alias("patient_id"), F.to_date("service_date").alias("day"), "ndc",
                  clean(F.col("rendering_npi")).alias("npi")))
mx_patients = mx_w.select("patient_id").distinct()

rx_w = (df["rx"].withColumn("product", ryb_product("ndc", "product_name"))
          .filter(F.col("product").isNotNull() & in_window("fill_date"))
          .select(F.concat_ws("|", "dv1", "dv2").alias("patient_id"), "claim_id", clean(F.col("prescriber_npi")).alias("npi"))
          .join(mx_patients, "patient_id", "left_anti"))           # patients already in MX are taken from MX

mx_npi = (mx_w.filter("npi IS NOT NULL").groupBy("npi")
              .agg(F.countDistinct("patient_id").alias("mx_patients"),
                   F.countDistinct(F.concat_ws("|", "patient_id", "day", "ndc")).alias("mx_infusions")))
rx_npi = (rx_w.filter("npi IS NOT NULL").groupBy("npi")
              .agg(F.countDistinct("patient_id").alias("rx_patients"), F.countDistinct("claim_id").alias("rx_fills")))
writer_npi = (mx_npi.join(rx_npi, "npi", "full").fillna(0)
                    .join(npi_jnj, "npi", "left")
                    .withColumn("hcp_key", F.coalesce("jnj_id", F.concat(F.lit("NPI_"), "npi"))))

n = writer_npi.count(); m = writer_npi.filter("jnj_id IS NOT NULL").count()
print(f"MX rows in window: {mx_w.count():,} | RX rows in window (RX-only patients): {rx_w.count():,}")
print(f"writer NPIs: {n:,} | with J&J ID: {m:,} ({100*m/max(n,1):.1f}%)")

writers = (writer_npi.groupBy("hcp_key")
           .agg(F.concat_ws(";", F.sort_array(F.collect_set("npi"))).alias("writer_npis"),
                F.first("dmn_specialty", True).alias("dmn_specialty"),
                F.sum("mx_patients").alias("mx_patients"), F.sum("mx_infusions").alias("mx_infusions"),
                F.sum("rx_patients").alias("rx_patients"), F.sum("rx_fills").alias("rx_fills"))
           .withColumn("ryb_patients_52w", F.col("mx_patients") + F.col("rx_patients"))
           .withColumn("is_ryb_writer", F.lit(True))
           .cache())
print("Rybrevant writer HCPs (52 weeks):", f"{writers.count():,}")

## 6. McKesson EMR (extra HCP-level source)

### What this cell does: McKesson HCPs and how they overlap with the other two lists
- McKesson alerts are Rybrevant treatment triggers (Ryb IV / Faspro + chemo, TKI or Lazcluze), with the physician NPI.
- Shows the alert types and diagnoses, all-time and in the window (by `diagnosis_date`).
- Physician NPI → J&J ID through dmn_cust, rolled up to one row per HCP.
- **Overlap:** McKesson HCPs that are also Rybrevant writers in claims, also in the H&N cohort, in both, or in neither.
- `INCLUDE_MCKESSON_AS_WRITER` (section 1) decides whether McKesson-only HCPs count as Rybrevant writers in the segments. Default **False**, pending a team decision.

In [ ]:
mck = df["mckesson"]
mck.groupBy("alert_name").count().orderBy(F.desc("count")).show(truncate=False)
mck.groupBy("diagnosis").count().orderBy(F.desc("count")).show(truncate=False)

mck_w = (mck.filter(in_window("diagnosis_date"))
            .select(clean(F.col("physician_npi")).alias("npi"), "alert_name", "alert_seq_num")
            .filter("npi IS NOT NULL"))
print(f"McKesson rows: all {mck.count():,} | in window {mck_w.count():,}")

mck_hcp = (mck_w.join(npi_jnj.select("npi", "jnj_id"), "npi", "left")
                .withColumn("hcp_key", F.coalesce("jnj_id", F.concat(F.lit("NPI_"), "npi")))
                .groupBy("hcp_key").agg(F.countDistinct("alert_seq_num").alias("mck_alerts"),
                                        F.concat_ws(";", F.sort_array(F.collect_set("alert_name"))).alias("mck_alert_types"))
                .withColumn("in_mckesson", F.lit(True))
                .cache())

(mck_hcp.join(writers.select("hcp_key", "is_ryb_writer"), "hcp_key", "left")
        .join(cohort_hcp.select("hcp_key", "in_hn_cohort"), "hcp_key", "left")
        .groupBy(F.coalesce("is_ryb_writer", F.lit(False)).alias("also_ryb_writer_in_claims"),
                 F.coalesce("in_hn_cohort", F.lit(False)).alias("also_in_hn_cohort"))
        .count().orderBy("also_ryb_writer_in_claims", "also_in_hn_cohort").show())

## 7. Xponent retail sales (HCP level)

### What this cell does: Xponent prescribers and how they overlap with the other lists
- Xponent = retail pharmacy sales per prescribing HCP (L0): `jnj_id` is the **HCP's J&J ID**, `equ_trx` = equivalized prescriptions.
- Keeps Rybrevant / Rybrevant Faspro, `caln_dt` in the window. Shows the products and TRx.
- **Where do the Xponent IDs sit in OPAL?** Normally column `0` (HCP). IDs not found there can't be rolled up to an account.
- One row per HCP: TRx in the window.
- **Overlap:** Xponent HCPs that are also Rybrevant writers in claims, also in the H&N cohort, in both, or in neither.
- `INCLUDE_XPONENT_AS_WRITER` (section 1) decides whether Xponent-only HCPs count as writers. Default **False**, pending a team decision.

In [ ]:
xp = (df["xpon"].filter(F.upper("prod_nm").contains("RYBREVANT") & in_window("caln_dt"))
               .select(clean(F.col("jnj_id")).alias("hcp_key"), F.upper(F.trim("prod_nm")).alias("product"),
                       F.col("equ_trx").cast("double").alias("trx")))
xp.groupBy("product").agg(F.count("*").alias("rows"), F.countDistinct("hcp_key").alias("hcps"),
                          F.round(F.sum("trx"), 2).alias("equ_trx")).show(truncate=False)

opal_hcp_ids = df["opal"].select(clean(F.col("`0`")).alias("hcp_key")).distinct().withColumn("in_opal_col0", F.lit(True))
xpon_hcp = (xp.groupBy("hcp_key").agg(F.round(F.sum("trx"), 4).alias("xpon_trx_52w"))
              .join(opal_hcp_ids, "hcp_key", "left").fillna(False, subset=["in_opal_col0"])
              .withColumn("in_xponent", F.lit(True))
              .cache())
(xpon_hcp.groupBy("in_opal_col0").agg(F.count("*").alias("hcps"), F.round(F.sum("xpon_trx_52w"), 2).alias("equ_trx"))
         .show())

(xpon_hcp.join(writers.select("hcp_key", "is_ryb_writer"), "hcp_key", "left")
         .join(cohort_hcp.select("hcp_key", "in_hn_cohort"), "hcp_key", "left")
         .groupBy(F.coalesce("is_ryb_writer", F.lit(False)).alias("also_ryb_writer_in_claims"),
                  F.coalesce("in_hn_cohort", F.lit(False)).alias("also_in_hn_cohort"))
         .agg(F.count("*").alias("hcps"), F.round(F.sum("xpon_trx_52w"), 2).alias("equ_trx"))
         .orderBy("also_ryb_writer_in_claims", "also_in_hn_cohort").show())

## 8. Segments

### What this cell does: create the three segments
- Puts all HCPs from the H&N cohort and the Rybrevant writer list side by side (full outer join on the HCP key).
  - Adds McKesson writers too if `INCLUDE_MCKESSON_AS_WRITER = True`.
- **Segment:**
  - in cohort and a writer → **H&N + Lung**;
  - cohort only → **H&N only**;
  - writer only → **Lung / Rybrevant only**.
- Specialty = the cohort's `Final Specialty`, else dmn_cust's `prim_spclt_nm`.
- Shows HCP counts and % per segment.
- Also shows the segment counts for every choice of extra writer sources (claims only, + McKesson, + Xponent, + both).

In [ ]:
def build_segments(include_mck, include_xp):
    w = writers.select("hcp_key", "is_ryb_writer")
    if include_mck:
        w = w.unionByName(mck_hcp.select("hcp_key", F.lit(True).alias("is_ryb_writer")))
    if include_xp:
        w = w.unionByName(xpon_hcp.select("hcp_key", F.lit(True).alias("is_ryb_writer")))
    w = w.distinct()
    s = (cohort_hcp.select("hcp_key", "in_hn_cohort").join(w, "hcp_key", "full")
         .fillna(False, subset=["in_hn_cohort", "is_ryb_writer"])
         .withColumn("segment", F.when(F.col("in_hn_cohort") & F.col("is_ryb_writer"), "H&N + Lung")
                                 .when(F.col("in_hn_cohort"), "H&N only")
                                 .otherwise("Lung / Rybrevant only")))
    return s

def show_counts(s, title):
    print(title)
    (s.groupBy("segment").count()
      .withColumn("pct", F.round(100 * F.col("count") / F.sum("count").over(Window.partitionBy()), 1))
      .orderBy("segment").show(truncate=False))

seg = build_segments(INCLUDE_MCKESSON_AS_WRITER, INCLUDE_XPONENT_AS_WRITER)
hcp_segments = (seg.join(cohort_hcp.drop("in_hn_cohort"), "hcp_key", "left")
                   .join(writers.drop("is_ryb_writer"), "hcp_key", "left")
                   .join(mck_hcp, "hcp_key", "left")
                   .join(xpon_hcp.drop("in_opal_col0"), "hcp_key", "left")
                   .fillna(False, subset=["in_mckesson", "in_xponent"])
                   .fillna(0, subset=["mx_patients", "mx_infusions", "rx_patients", "rx_fills", "ryb_patients_52w",
                                      "mck_alerts", "xpon_trx_52w"])
                   .withColumn("specialty", F.coalesce("cohort_specialty", "dmn_specialty"))
                   .withColumn("jnj_id", F.when(~F.col("hcp_key").startswith("NPI_"), F.col("hcp_key")))
                   .cache())
show_counts(hcp_segments, f"Segments used (McKesson as writer: {INCLUDE_MCKESSON_AS_WRITER}, Xponent as writer: {INCLUDE_XPONENT_AS_WRITER})")

print("Comparison: HCPs per segment for every choice of extra writer sources")
combos = None
for mck_on in (False, True):
    for xp_on in (False, True):
        t = (build_segments(mck_on, xp_on).groupBy("segment").count()
             .withColumn("writers_from", F.lit("claims" + (" + McKesson" if mck_on else "") + (" + Xponent" if xp_on else ""))))
        combos = t if combos is None else combos.unionByName(t)
combos.groupBy("writers_from").pivot("segment", ["H&N only", "H&N + Lung", "Lung / Rybrevant only"]).sum("count").show(truncate=False)
print("HCPs without a J&J ID (kept by NPI):", hcp_segments.filter("jnj_id IS NULL").count())

### What this cell does: describe the segments
- **By specialty:** top 25 specialties, HCP counts per segment.
- **By practice setting:** H&N cohort HCPs only, since the setting comes from the cohort table.
- **Volumes per segment:**
  - Rybrevant patients and infusions (last 52 weeks, writers), and Xponent TRx;
  - H&N treating volume (2 years, cohort).
- **Look at:** whether "H&N + Lung" HCPs are mostly hem/onc specialists, and how much Rybrevant volume they carry.

In [ ]:
segs = ["H&N only", "H&N + Lung", "Lung / Rybrevant only"]
(hcp_segments.groupBy(F.coalesce("specialty", F.lit("UNKNOWN")).alias("specialty"))
             .pivot("segment", segs).count().fillna(0)
             .withColumn("total", sum(F.col(f"`{s}`") for s in segs))
             .orderBy(F.desc("total")).show(25, False))
(hcp_segments.filter("in_hn_cohort").groupBy(F.coalesce("practice_setting", F.lit("UNKNOWN")).alias("practice_setting"))
             .pivot("segment", segs[:2]).count().fillna(0).orderBy(F.desc("`H&N only`")).show(25, False))
(hcp_segments.groupBy("segment")
             .agg(F.count("*").alias("hcps"), F.sum("ryb_patients_52w").alias("ryb_patients_52w"),
                  F.sum("mx_infusions").alias("ryb_infusions_52w"), F.round(F.sum("xpon_trx_52w"), 2).alias("xpon_trx_52w"),
                  F.sum("hn_treating_pts_2y").alias("hn_treating_pts_2y"))
             .orderBy("segment").show(truncate=False))

## 9. Accounts: HCP segments + DDD and Xponent sales at L2/L3

### What this cell does: put each HCP on its L2/L3 account (OPAL)
- HCP J&J ID → OPAL column `0` → `L2_L3`. This is the same rule for every HCP (L2 when available, otherwise L3).
- **Check against the cohort:** the cohort has its own `Mapped Account JNJ ID`, which can be L2, L3 or L5.
  - It shows how often OPAL's `L2_L3` matches one of the cohort's mapped accounts.
- Shows how many HCPs in each segment reach an account.

In [ ]:
def txt(col):
    s = F.trim(F.col(f"`{col}`").cast("string"))
    return F.when(s != "", s)

o = df["opal"].select(txt("0").alias("opal_hcp"), txt("1").alias("l1"), txt("2").alias("l2"), txt("3").alias("l3"),
                      txt("L2_L3").alias("account_id"))
hcp_acct = (hcp_segments.filter("jnj_id IS NOT NULL").select("hcp_key", "jnj_id", "segment")
            .join(o.select(F.col("opal_hcp").alias("jnj_id"), "account_id").filter("account_id IS NOT NULL"), "jnj_id", "left")
            .cache())
(hcp_acct.groupBy("segment").agg(F.countDistinct("hcp_key").alias("hcps_with_jnj_id"),
                                 F.countDistinct(F.when(F.col("account_id").isNotNull(), F.col("hcp_key"))).alias("reach_L2_L3"))
         .orderBy("segment").show(truncate=False))

coh_acc = coh_k.select("hcp_key", "cohort_account_id").distinct()
cmp = (hcp_acct.filter("account_id IS NOT NULL").join(coh_acc, "hcp_key")
               .groupBy("hcp_key").agg(F.max((F.col("account_id") == F.col("cohort_account_id")).cast("int")).alias("match")))
print("cohort HCPs whose OPAL L2_L3 equals one of their cohort mapped accounts:",
      cmp.filter("match = 1").count(), "of", cmp.count())

### What this cell does: roll DDD and Xponent up to L2/L3 and combine them with the HCP mix
- **DDD in the window**, Rybrevant + Faspro, mapped to an account:
  - the DDD `jnj_id` is an L1 outlet in OPAL column `1` → its `L2_L3`;
  - or the ID is itself an L2 / L3 account.
  - DDD IDs not found in OPAL stay unmapped (open question Q9).
- **Xponent in the window**, Rybrevant + Faspro: the HCP's `jnj_id` → OPAL column `0` → `L2_L3`. Prints how much TRx reached an account.
- **Per account:**
  - DDD units (non-retail) and Xponent TRx (retail), kept in **separate columns** because they are different measures;
  - the number of HCPs in each segment.
- **Account type:**
  - **has H&N HCPs**: at least one H&N only or H&N + Lung HCP;
  - **Lung writers only**;
  - **no linked HCPs**.
- Shows the share of DDD units and of Xponent TRx in each account type, then the top 25 accounts.

In [ ]:
ddd = (df["ddd"].filter(F.upper("prod_nm").contains("RYBREVANT") & in_window("caln_dt"))
                .select(F.col("jnj_id").cast("string").alias("ddd_id"), F.upper(F.trim("prod_nm")).alias("product"),
                        F.col("equiv_uns").cast("double").alias("units")))
via_l1 = (o.filter("l1 IS NOT NULL AND account_id IS NOT NULL").groupBy(F.col("l1").alias("ddd_id"))
           .agg(F.first("account_id").alias("account_id"), F.countDistinct("account_id").alias("n_acc")))
via_l2 = o.filter("l2 IS NOT NULL").select(F.col("l2").alias("ddd_id"), F.col("l2").alias("acc2")).distinct()
via_l3 = o.filter("l3 IS NOT NULL").select(F.col("l3").alias("ddd_id"), F.col("l3").alias("acc3")).distinct()
ddd_m = (ddd.join(via_l1, "ddd_id", "left").join(via_l2, "ddd_id", "left").join(via_l3, "ddd_id", "left")
            .withColumn("account_id", F.coalesce("account_id", "acc2", "acc3")))
print("DDD units in window:", ddd_m.agg(F.round(F.sum("units"), 1)).collect()[0][0],
      "| mapped to an account:", ddd_m.filter("account_id IS NOT NULL").agg(F.round(F.sum("units"), 1)).collect()[0][0],
      "| outlets mapping to >1 account:", via_l1.filter("n_acc > 1").count())

acct_sales = (ddd_m.filter("account_id IS NOT NULL").groupBy("account_id")
              .pivot("product", ["RYBREVANT", "RYBREVANT FASPRO"]).agg(F.round(F.sum("units"), 3)).fillna(0)
              .withColumn("ddd_units", F.col("RYBREVANT") + F.col("`RYBREVANT FASPRO`")))
# Xponent (L0 HCP) -> L2/L3 through OPAL column 0
xp_acct = xp.join(o.select(F.col("opal_hcp").alias("hcp_key"), "account_id"), "hcp_key", "left")
print("Xponent TRx in window:", xp_acct.agg(F.round(F.sum("trx"), 2)).collect()[0][0],
      "| mapped to an account:", xp_acct.filter("account_id IS NOT NULL").agg(F.round(F.sum("trx"), 2)).collect()[0][0])
acct_xpon = (xp_acct.filter("account_id IS NOT NULL").groupBy("account_id")
             .agg(F.round(F.sum("trx"), 3).alias("xpon_trx"), F.countDistinct("hcp_key").alias("xpon_hcps")))

acct_mix = (hcp_acct.filter("account_id IS NOT NULL").groupBy("account_id")
            .pivot("segment", segs).agg(F.countDistinct("hcp_key")).fillna(0))
accounts = (acct_sales.join(acct_xpon, "account_id", "full").join(acct_mix, "account_id", "full").fillna(0)
            .withColumn("account_type",
                        F.when((F.col("`H&N only`") + F.col("`H&N + Lung`")) > 0, "has H&N HCPs")
                         .when(F.col("`Lung / Rybrevant only`") > 0, "Lung writers only")
                         .otherwise("no linked HCPs")))
(accounts.groupBy("account_type").agg(F.count("*").alias("accounts"), F.round(F.sum("ddd_units"), 1).alias("ddd_units"),
                                      F.round(F.sum("xpon_trx"), 1).alias("xpon_trx"))
         .withColumn("pct_ddd_units", F.round(100 * F.col("ddd_units") / F.sum("ddd_units").over(Window.partitionBy()), 1))
         .withColumn("pct_xpon_trx", F.round(100 * F.col("xpon_trx") / F.sum("xpon_trx").over(Window.partitionBy()), 1))
         .orderBy(F.desc("ddd_units")).show(truncate=False))
accounts.orderBy(F.desc("ddd_units")).show(25, False)

## 10. Save

### What this cell does: optionally save the results to Dataiku
- Leave `SAVE = False` to only look at the results.
- To save: create empty output datasets in the Flow with these names, then set `SAVE = True`.
  - `HN_HCP_SEGMENTS`: one row per HCP with its segment, IDs, specialty and volumes.
  - `HN_ACCOUNT_SEGMENT_MIX`: one row per L2/L3 account with DDD units, Xponent TRx and HCP counts per segment.

In [ ]:
SAVE = False
outputs = {"HN_HCP_SEGMENTS": hcp_segments, "HN_ACCOUNT_SEGMENT_MIX": accounts}
if SAVE:
    for name, d in outputs.items():
        dkuspark.write_with_schema(dataiku.Dataset(name), d)
        print("saved", name)